# LightGBM 하이퍼파라미터 튜닝 (Optuna + Early Stopping)

main.py와 동일한 전처리 파이프라인을 사용하여, Optuna로 LightGBM의 하이퍼파라미터를 탐색합니다.
기존 실험(기본값 LGBMClassifier)이 0.739 근처에서 수렴했던 것과 달리, 이번에는:

1. `n_estimators`를 크게(최대 3000) 주고 `early_stopping`으로 적정 시점에서 멈춤 — 기존 실험은 사실상 underfitting 상태였을 수 있음
2. `num_leaves`, `max_depth` 등 트리 복잡도 관련 파라미터의 탐색 범위를 넓힘
3. 150 trial로 충분히 탐색 (TPE 샘플러는 80~150 trial 사이에서 수익이 급격히 줄어드는 경향이 있어, 시간 대비 가장 합리적인 지점)

**구성**
1. 튜닝 설정 및 데이터 전처리
2. Optuna 탐색 실행 (시간 오래 걸릴 수 있음 — 중간에 중단해도 결과 보존됨)
3. 튜닝된 파라미터로 최종 5-Fold 학습
4. 제출 파일 생성 (버전 차수 관리 포함)

**사전 준비**: `uv add optuna` 가 안 되어 있다면 터미널에서 먼저 실행하세요.


## 1. 라이브러리 및 설정

In [ ]:
import json
import os
import time

import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "../data"
SUBMISSION_DIR = "../submission file"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
SUBMISSION_PATH = f"{DATA_DIR}/sample_submission.csv"

TARGET_COL = "임신 성공 여부"
ID_COL = "ID"
RANDOM_STATE = 1004
N_SPLITS = 5

# --- 튜닝 설정 ---
N_TRIALS = 150  # 처음 5~10 trial 실행 시간을 보고 너무 길면 줄이세요 (50~80도 충분할 수 있음)
N_FOLDS_TUNING = 3  # 튜닝 단계는 3-fold로 속도 확보 (최종 학습은 5-fold)
EARLY_STOPPING_ROUNDS = 50

STUDY_DB = "sqlite:///optuna_lgbm.db"
STUDY_NAME = "lgbm_tuning_v2"
BEST_PARAMS_PATH = "best_params.json"
VERSION_FILE = "version_state.json"
MODEL_NAME = "LGBM_tuned"

## 2. 버전(차수) 관리

기존 `submission.ipynb`와 동일한 규칙입니다 — 제출하지 않고 실험만 반복하면 같은 차수가 유지되고,
실제로 제출 완료 표시를 한 다음부터만 차수가 올라갑니다. `version_state.json`을 공유하므로
`submission.ipynb`와 차수가 함께 관리됩니다.


In [8]:
def load_version_state():
    if os.path.exists(VERSION_FILE):
        with open(VERSION_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"version": 1, "submitted": False}


def save_version_state(state):
    with open(VERSION_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def get_active_version():
    state = load_version_state()
    if state["submitted"]:
        state["version"] += 1
        state["submitted"] = False
        save_version_state(state)
    return state["version"]


state = load_version_state()
print("현재 차수:", state["version"], "차  (제출 완료 여부:", state["submitted"], ")")

현재 차수: 1 차  (제출 완료 여부: False )


## 3. 데이터 로드 및 전처리 (main.py와 동일한 파이프라인)

In [9]:
def preprocess(train, test):
    test_ids = test[ID_COL].copy()
    train = train.drop(columns=[ID_COL])
    test = test.drop(columns=[ID_COL])

    # 구조적 결측 플래그 (DI 전체 결측 / 동결여부 조건부 결측)
    for df in (train, test):
        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
        df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    # 정보량 없는 컬럼 제거
    cols_to_drop = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
    train = train.drop(columns=cols_to_drop)
    test = test.drop(columns=cols_to_drop)

    # 결측치 처리: 수치형 -1, 범주형 "해당없음"
    categorical_cols = train.select_dtypes(include=["object", "str"]).columns
    numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns
    na_cols = train.isna().sum().loc[lambda x: x > 0].index

    for col in na_cols:
        if col in categorical_cols:
            train[col] = train[col].fillna("해당없음")
            test[col] = test[col].fillna("해당없음")
        elif col in numerical_cols:
            train[col] = train[col].fillna(-1)
            test[col] = test[col].fillna(-1)

    # 라벨 인코딩 (test에만 있는 라벨 처리)
    for col in categorical_cols:
        le = LabelEncoder()
        le.fit(train[col])
        unseen_labels = set(test[col].unique()) - set(le.classes_)
        if unseen_labels:
            le.classes_ = np.append(le.classes_, list(unseen_labels))
        train[col] = le.transform(train[col])
        test[col] = le.transform(test[col])

    return train, test, test_ids


train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_p, test_p, test_ids = preprocess(train_raw, test_raw)

X = train_p.drop(columns=[TARGET_COL])
y = train_p[TARGET_COL]
print(f"전처리 완료: train={X.shape}, test={test_p.shape}")

전처리 완료: train=(256351, 67), test=(90067, 67)


## 4. Optuna 탐색 실행

`n_estimators`를 크게 주고 early stopping으로 적정 시점에서 멈추는 방식입니다.
**시간이 오래 걸릴 수 있습니다** — 처음 5~10 trial이 끝나는 데 걸린 시간을 보고
전체 소요 시간을 가늠해보세요. 너무 길면 커널을 중단(Interrupt)해도 됩니다.
`best_params.json`에 그 시점까지의 최선 결과가 저장되어 있어서, 이 셀을 다시
실행하면 중단된 지점부터 이어서 탐색합니다.


In [10]:
def make_objective(X, y):
    skf = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            "random_state": RANDOM_STATE,
            "verbose": -1,
            "n_estimators": trial.suggest_int("n_estimators", 500, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 150),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        }

        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            X_tr2, X_es, y_tr2, y_es = train_test_split(
                X_tr, y_tr, test_size=0.15, random_state=RANDOM_STATE, stratify=y_tr
            )

            model = LGBMClassifier(**params)
            model.fit(
                X_tr2,
                y_tr2,
                eval_set=[(X_es, y_es)],
                eval_metric="auc",
                callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
            )

            proba = model.predict_proba(X_val)[:, 1]
            scores.append(roc_auc_score(y_val, proba))

        return float(np.mean(scores))

    return objective


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=STUDY_DB,
    study_name=STUDY_NAME,
    load_if_exists=True,
)

t0 = time.time()


def callback(study, trial):
    elapsed = time.time() - t0
    print(
        f"[{elapsed:6.1f}s] trial {trial.number:>3}: "
        f"{trial.value:.5f}  (best: {study.best_value:.5f})"
    )
    with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {"best_value": study.best_value, "best_params": study.best_params},
            f,
            ensure_ascii=False,
            indent=2,
        )


print(f"Optuna 탐색 시작: {N_TRIALS} trials, {N_FOLDS_TUNING}-fold CV, early stopping={EARLY_STOPPING_ROUNDS}\n")

try:
    study.optimize(make_objective(X, y), n_trials=N_TRIALS, callbacks=[callback])
except KeyboardInterrupt:
    print("\n사용자에 의해 중단됨. 지금까지의 결과를 저장합니다.")

print(f"\n탐색 완료 (총 {len(study.trials)} trials, {time.time() - t0:.1f}s)")
print(f"Best score (3-fold): {study.best_value:.5f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-06-21 01:02:12,199] A new study created in RDB with name: lgbm_tuning_v2


Optuna 탐색 시작: 150 trials, 3-fold CV, early stopping=50



[I 2026-06-21 01:02:22,794] Trial 0 finished with value: 0.7373590859789348 and parameters: {'n_estimators': 597, 'learning_rate': 0.06505741597355452, 'num_leaves': 194, 'max_depth': 14, 'min_child_samples': 86, 'subsample': 0.5402415367770963, 'colsample_bytree': 0.6158181492470156, 'reg_alpha': 0.0016267473224230488, 'reg_lambda': 0.11953637030473808, 'min_split_gain': 0.6071090928914465}. Best is trial 0 with value: 0.7373590859789348.


[  10.6s] trial   0: 0.73736  (best: 0.73736)


[I 2026-06-21 01:02:51,578] Trial 1 finished with value: 0.7390857333139046 and parameters: {'n_estimators': 760, 'learning_rate': 0.01790651364770632, 'num_leaves': 172, 'max_depth': 14, 'min_child_samples': 69, 'subsample': 0.8568956619166972, 'colsample_bytree': 0.5294310341477355, 'reg_alpha': 3.519074558947365, 'reg_lambda': 0.040652247361558605, 'min_split_gain': 0.7274294965586153}. Best is trial 1 with value: 0.7390857333139046.


...

[2818.3s] trial 147: 0.73910  (best: 0.73976)


[I 2026-06-21 01:49:29,768] Trial 148 finished with value: 0.7395368227516167 and parameters: {'n_estimators': 1638, 'learning_rate': 0.010889554482142445, 'num_leaves': 152, 'max_depth': 4, 'min_child_samples': 39, 'subsample': 0.8895042616324385, 'colsample_bytree': 0.7412791513221499, 'reg_alpha': 9.829563100056713, 'reg_lambda': 0.003058506615961935, 'min_split_gain': 0.33915923984682034}. Best is trial 112 with value: 0.7397625659955774.


[2837.6s] trial 148: 0.73954  (best: 0.73976)


[I 2026-06-21 01:49:44,841] Trial 149 finished with value: 0.7396061487317778 and parameters: {'n_estimators': 1599, 'learning_rate': 0.015410743446893563, 'num_leaves': 184, 'max_depth': 5, 'min_child_samples': 49, 'subsample': 0.9319116334288728, 'colsample_bytree': 0.7747331439546142, 'reg_alpha': 3.78318152814723, 'reg_lambda': 0.019927416371374698, 'min_split_gain': 0.04099399859362825}. Best is trial 112 with value: 0.7397625659955774.


[2852.7s] trial 149: 0.73961  (best: 0.73976)

탐색 완료 (총 150 trials, 2852.7s)
Best score (3-fold): 0.73976
Best params:
  n_estimators: 2357
  learning_rate: 0.013668973845707234
  num_leaves: 159
  max_depth: 5
  min_child_samples: 83
  subsample: 0.908634361540321
  colsample_bytree: 0.7944647062780377
  reg_alpha: 6.529095336314542
  reg_lambda: 0.00029237017648296726
  min_split_gain: 0.4557413392057567


## 5. 튜닝된 파라미터로 최종 5-Fold 학습

`best_params.json`을 읽어서, main.py와 동일한 Stratified 5-Fold 앙상블 구조로
최종 모델을 학습합니다.


In [11]:
with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    tuned = json.load(f)
best_params = tuned["best_params"]
print(f"튜닝된 파라미터 로드 (튜닝 단계 3-fold 점수: {tuned['best_value']:.5f})")
print(best_params)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
models = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, **best_params)
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, val_proba)
    fold_scores.append(score)
    models.append(model)
    print(f"[Fold {fold}/{N_SPLITS}] ROC-AUC: {score:.4f}")

best_local_score = float(np.mean(fold_scores))
print(f"\n5-Fold 평균 ROC-AUC: {best_local_score:.4f} (std: {np.std(fold_scores):.4f})")

튜닝된 파라미터 로드 (튜닝 단계 3-fold 점수: 0.73976)
{'n_estimators': 2357, 'learning_rate': 0.013668973845707234, 'num_leaves': 159, 'max_depth': 5, 'min_child_samples': 83, 'subsample': 0.908634361540321, 'colsample_bytree': 0.7944647062780377, 'reg_alpha': 6.529095336314542, 'reg_lambda': 0.00029237017648296726, 'min_split_gain': 0.4557413392057567}
[Fold 1/5] ROC-AUC: 0.7440
[Fold 2/5] ROC-AUC: 0.7404
[Fold 3/5] ROC-AUC: 0.7380
[Fold 4/5] ROC-AUC: 0.7385
[Fold 5/5] ROC-AUC: 0.7393

5-Fold 평균 ROC-AUC: 0.7400 (std: 0.0021)


## 6. 테스트 예측 및 제출 파일 생성

In [ ]:
final_pred = np.mean([model.predict_proba(test_p)[:, 1] for model in models], axis=0)

VERSION = get_active_version()
LOCAL_SCORE = f"{best_local_score:.5f}"
SAVE_NAME = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->).csv"
SAVE_PATH = f"{SUBMISSION_DIR}/{SAVE_NAME}"

submission = pd.read_csv(SUBMISSION_PATH)
submission["ID"] = test_ids.values
submission["probability"] = final_pred
submission.to_csv(SAVE_PATH, index=False)

print(f"저장 완료: {SAVE_NAME}")
print(submission.head())

저장 완료: 1차_LGBM_tuned_submit_improved(0.74002->).csv
           ID  probability
0  TEST_00000     0.001323
1  TEST_00001     0.002195
2  TEST_00002     0.152920
3  TEST_00003     0.106357
4  TEST_00004     0.511431


## 7. 실제 대회에 제출한 뒤 (점수 확인 후 실행)

**대회 사이트에 위에서 만든 csv를 업로드해서 실제 점수를 받은 뒤에만** 이 셀을 실행하세요.


In [ ]:
ACTUAL_SCORE = "0.0000"  # 대회에서 받은 실제 점수를 여기에 입력하세요

old_path = SAVE_PATH
new_name = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->{ACTUAL_SCORE}).csv"
new_path = f"{DATA_DIR}/{new_name}"

os.rename(old_path, new_path)
print(f"파일명 변경 완료: {new_name}")

state = load_version_state()
state["submitted"] = True
save_version_state(state)
print(f"{VERSION}차 제출 완료로 표시됨. 다음 실행부터는 {VERSION + 1}차로 시작됩니다.")